In [ ]:
import pandas as pd
import zipfile


In [2]:
# run once if pyarrow is missing:  %pip install pyarrow
import collections
import pandas as pd

df = pd.read_parquet("../data/NQ/nq_open_test.parquet")

print(df.shape, list(df.columns))
display(df.head(15))

for _, r in df.head(5).iterrows():
    print(f"Q: {r.question}")
    print(f"A: {list(r.answer)}\n")

n = df.answer.map(len)
w = [len(s.split()) for a in df.answer for s in a]

print("spans/question:", dict(sorted(collections.Counter(n).items())[:6]))
print(f"multi-span: {(n > 1).sum()}/{len(df)} = {(n > 1).mean():.1%}   max spans: {n.max()}")
print(f"answer words: mean {sum(w) / len(w):.2f}, max {max(w)}")

(3610, 2) ['question', 'answer']


,question,answer
0,when was the last time anyone was on the moon,"[14 December 1972 UTC, December 1972]"
1,who wrote he ain't heavy he's my brother lyrics,"[Bobby Scott, Bob Russell]"
2,how many seasons of the bastard executioner ar...,"[one, one season]"
3,when did the eagles win last super bowl,[2017]
4,who won last year's ncaa women's basketball,[South Carolina]
5,when did the isle of wight become an island,[During the last Ice Age]
6,love yourself by justin bieber is about who,[Rihanna]
7,who was the ruler of england in 1616,[James I]
8,what is the hot coffee mod in san andreas,[a normally inaccessible mini-game]
9,what is the maximum data rate for the 802.11a ...,[54 Mbit/s]


Q: when was the last time anyone was on the moon
A: ['14 December 1972 UTC', 'December 1972']

Q: who wrote he ain't heavy he's my brother lyrics
A: ['Bobby Scott', 'Bob Russell']

Q: how many seasons of the bastard executioner are there
A: ['one', 'one season']

Q: when did the eagles win last super bowl
A: ['2017']

Q: who won last year's ncaa women's basketball
A: ['South Carolina']

spans/question: {1: 2076, 2: 953, 3: 306, 4: 118, 5: 58, 6: 28}
multi-span: 1534/3610 = 42.5%   max spans: 23
answer words: mean 2.16, max 6


In [11]:
import csv

path = "../data/NQ/Natural-Questions-Filtered.csv"

counts = {"total": 0, "short_only": 0, "long_only": 0, "both": 0, "neither": 0}

with open(path, encoding="utf-8") as f:
    for row in csv.DictReader(f):          # sep="\t" -> add delimiter="\t"
        s = bool(row["short_answers"].strip())
        l = bool(row["long_answers"].strip())
        counts["total"] += 1
        counts["both" if s and l else "short_only" if s else "long_only" if l else "neither"] += 1

counts["any_short"] = counts["short_only"] + counts["both"]
counts["any_long"] = counts["long_only"] + counts["both"]
print(counts)

{'total': 86212, 'short_only': 0, 'long_only': 0, 'both': 86212, 'neither': 0, 'any_short': 86212, 'any_long': 86212}


In [13]:
import re
import pandas as pd

PATH = "../data/NQ/Natural-Questions-Filtered.csv"
SPLIT = re.compile(r",\s+(?=[A-Z0-9])")   # the join signature from caveat 19

df = pd.read_csv(PATH)
s = df["short_answers"].fillna("")
n = len(df)

has_comma  = s.str.contains(",", regex=False)
looks_join = s.apply(lambda a: bool(SPLIT.search(a)))
inner_only = has_comma & ~looks_join

print(f"rows                             : {n:,}")
print(f"short answers containing a comma : {has_comma.sum():,}  ({has_comma.mean():.1%})")
print(f"  matching the multi-span join   : {looks_join.sum():,}  ({looks_join.mean():.1%})")
print(f"  comma but NOT a join           : {inner_only.sum():,}  ({inner_only.mean():.1%})")

spans = s[looks_join].apply(lambda a: len(SPLIT.split(a)))
print(f"  spans per joined answer        : mean {spans.mean():.2f}  max {spans.max()}")

rows                             : 86,212
short answers containing a comma : 19,423  (22.5%)
  matching the multi-span join   : 18,463  (21.4%)
  comma but NOT a join           : 960  (1.1%)
  spans per joined answer        : mean 2.57  max 21


Manually check types of errors and fix as many as possible. (date format is one)


## Test corpus generation

In [3]:
import os
from pathlib import Path

# notebook may not start at repo root
if not Path("rag_pipeline/build_article_corpus.py").exists():
    os.chdir("..")
print("cwd:", Path.cwd())

from rag_pipeline.build_article_corpus import main

rc = main([
    "--input",  "../data/wikiDump/psgs_sample_500.tsv",
    "--output", "../data/wikiDump/articles_sample_500.jsonl",
    "--verify", "100000",          # verify every article; the sample only has 9
])
print("exit code:", rc, "->", {0: "clean", 1: "input not found", 2: "VERIFY FAILED"}[rc])

cwd: c:\Users\Jespe\Desktop\BACKUP_UNI\ITU\Masters\Thesis RAG Systems\RAGNAR
[articles] input not found: ../data/wikiDump/psgs_sample_500.tsv
exit code: 1 -> input not found


In [7]:
import json

recs = [json.loads(l) for l in open("data/wikiDump/articles_sample_500.jsonl", encoding="utf-8")]
print(f"{len(recs)} articles, keys: {list(recs[0])}\n")

for r in sorted(recs, key=lambda r: -r["n_passages"])[:3]:
    print(f"[{r['wikipedia_id']}] {r['wikipedia_title']}  ({r['n_passages']} passages, {len(r['text']):,} chars)")
    #print(f"  {r['text'][:200]}...\n")
    print(r["text"])
    break

# paragraph structure survived the join?
para = sum(r["text"].count("\n") for r in recs)
print(f"newlines (paragraph breaks) preserved: {para}")
print(f"title line stripped from lead passage: {not recs[0]['text'].startswith(recs[0]['wikipedia_title'])}")

9 articles, keys: ['wikipedia_id', 'wikipedia_title', 'text', 'n_passages']

[359] List of Atlas Shrugged characters  (43 passages, 22,315 chars)
This is a list of characters in Ayn Rand's novel "Atlas Shrugged."
Major characters.
The following are major characters from the novel.
Major characters Protagonists.
Major characters Protagonists Dagny Taggart.
Dagny Taggart is the protagonist of the novel. She is Vice-President in Charge of Operations for Taggart Transcontinental, under her brother, James Taggart. Given James' incompetence, Dagny is responsible for all the workings of the railroad.
Major characters Protagonists Francisco d'Anconia.
Francisco d'Anconia is one of the central characters in "Atlas Shrugged", an owner by inheritance of the world's largest copper mining operation. He is a childhood friend, and the first love, of Dagny Taggart. A child prodigy of exceptional talents, Francisco was dubbed the "climax" of the d'Anconia line, an already prestigious family of skilled 

### Chunker

In [17]:
import json
from rag_pipeline.components.chunker import (
    FixedWordChunker, SentenceChunker, ParagraphChunker,
)

recs  = [json.loads(l) for l in open("data/wikiDump/articles_sample_500.jsonl", encoding="utf-8")]
texts = [r["text"] for r in recs]
metas = [{"wikipedia_id": r["wikipedia_id"], "wikipedia_title": r["wikipedia_title"]} for r in recs]

chunker = FixedWordChunker(size=100, overlap=0, min_chunk_words=10)
chunks  = chunker.chunk_text(texts, metas)

print(repr(chunker))
print(f"{len(recs)} articles -> {len(chunks)} chunks")

w = [len(c.text.split()) for c in chunks]
print(f"words/chunk: mean {sum(w)/len(w):.1f}, min {min(w)}, max {max(w)}\n")

for c in chunks[:5]:
    print(f"[{c.chunk_id}] meta={c.metadata}")
    print(f"  {c.text}")
    #print(f"  {c.text[:120]}...\n")

FixedWordChunker(size=100, overlap=0, min_chunk_words=10)
9 articles -> 129 chunks
words/chunk: mean 97.8, min 40, max 100

[316#0] meta={'wikipedia_title': 'Academy Award for Best Production Design'}
  The Academy Award for Best Production Design recognizes achievement for art direction in film. The category's original name was Best Art Direction, but was changed to its current name in 2012 for the 85th Academy Awards. This change resulted from the Art Director's branch of the Academy of Motion Picture Arts and Sciences (AMPAS) being renamed the Designer's branch. Since 1947, the award is shared with the set decorator(s). It is awarded to the best interior design in a film. The films below are listed with their production year (for example, the 2000 Academy Award for Best Art Direction is
[316#1] meta={'wikipedia_title': 'Academy Award for Best Production Design'}
  given to a film from 1999). In the lists below, the winner of the award for each year is shown first, followed by the ot

# Check corrupt data full wiki

In [9]:
fullwiki = pd.read_csv("data/wikiDump/psgs_w100.tsv", sep="\t")
fullwiki.head()

KeyboardInterrupt: 

In [ ]:
fullwiki_nulls = fullwiki[fullwiki["text"].notna() & fullwiki["text"].str.strip().ne("")]
fullwiki_nulls

In [3]:
import csv

TARGET = 1_000_000
SRC = "data/wikiDump/psgs_w100.tsv"
DST = "data/wikiDump/wiki_1M.tsv"

with open(SRC, encoding="utf-8", errors="replace") as fin, \
     open(DST, "w", encoding="utf-8", newline="") as fout:
    reader = csv.DictReader(fin, delimiter="\t")
    writer = csv.DictWriter(fout, fieldnames=reader.fieldnames, delimiter="\t")
    writer.writeheader()
    for i, row in enumerate(reader):
        if i >= TARGET:
            break
        writer.writerow(row)

print(f"Done — {min(i+1, TARGET):,} passages written to {DST}")

Done — 1,000,000 passages written to data/wikiDump/wiki_1M.tsv


In [5]:
wiki1mil = pd.read_csv("data/wikiDump/wiki_1M.tsv", sep="\t")

wiki1mil.head()
wiki1mil.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 6 columns):
 #   Column           Non-Null Count    Dtype
---  ------           --------------    -----
 0   id               1000000 non-null  int64
 1   text             998692 non-null   str  
 2   wikipedia_title  1000000 non-null  str  
 3   wikipedia_id     1000000 non-null  int64
 4   end_paragraph    1000000 non-null  int64
 5   end_character    1000000 non-null  int64
dtypes: int64(4), str(2)
memory usage: 45.8 MB


In [ ]:
df = wiki1mil[wiki1mil["text"].notna() & wiki1mil["text"].str.strip().ne("")]
df.to_csv("data/wikiDump/wiki_1M_clean.tsv", sep="\t", index=False)


In [8]:
df = wiki1mil[wiki1mil["text"].isna() | wiki1mil["text"].str.strip().eq("")]
df

,id,text,wikipedia_title,wikipedia_id,end_paragraph,end_character
8074,8074,NaN,Daniel Chodowiecki,43655,19,38
14031,14031,NaN,"Ashfield, Massachusetts",116762,37,37
14715,14715,NaN,"Lawrence, Massachusetts",116748,346,83
18124,18124,NaN,Fairlight (group),621092,38,69
20290,20290,NaN,Alexandru Marghiloman,1214891,15,205
...,...,...,...,...,...,...
997952,997952,NaN,2016 in Scandinavian music,49414843,219,53
998246,998246,NaN,Selina Giles,49415983,8,311
998382,998382,NaN,Mark Spragg,49416623,20,67
998691,998691,NaN,Gud (music producer),49416870,21,36


In [ ]:
nq = pd.read_csv("data/NQ/Natural-Questions-Filtered.csv", sep=",")

nq.head()

,question,long_answers,short_answers
0,which is the most common use of opt-in e-mail ...,A common example of permission marketing is a ...,A newsletter sent to an advertising firm's cus...
1,how i.met your mother who is the mother,"Tracy McConnell, better known as `` The Mother...",Tracy McConnell
2,who had the most wins in the nfl,Active quarterback Tom Brady holds the records...,Tom Brady
3,who played mantis guardians of the galaxy 2,Pom Klementieff (born May 1986) is a French ac...,Pom Klementieff
4,the nashville sound brought a polished and cos...,"In the early 1960s, the Nashville sound began ...",The use of lush string arrangements with a rea...


In [30]:
nq.shape
nq.columns
nq.info()

<class 'pandas.DataFrame'>
RangeIndex: 86212 entries, 0 to 86211
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   question       86212 non-null  str  
 1   long_answers   86212 non-null  str  
 2   short_answers  86212 non-null  str  
dtypes: str(3)
memory usage: 2.0 MB


In [4]:
hardware = pd.read_csv("results/hardware.csv", sep=",")

hardware.head()

,timestamp,disk_r_s,disk_w_s,disk_rkB_s,disk_wkB_s,disk_r_await_ms,disk_w_await_ms,disk_util_pct,gpu_mem_used_mb,gpu_mem_total_mb,gpu_mem_free_mb,gpu_util_pct
0,2026-04-29T01:51:36,NaN,NaN,NaN,NaN,NaN,NaN,NaN,692.0,8192.0,7368.0,1.0
1,2026-04-29T01:51:37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,698.0,8192.0,7362.0,0.0
2,2026-04-29T01:51:39,NaN,NaN,NaN,NaN,NaN,NaN,NaN,714.0,8192.0,7346.0,0.0
3,2026-04-29T01:51:40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,714.0,8192.0,7346.0,0.0
4,2026-04-29T01:51:42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,719.0,8192.0,7341.0,8.0
